In [68]:
# importation de la bibliotheque pandas pour manipuler les donnees

import pandas as pd


In [69]:
# chargement du fichier CSV reduit contenant les recettes

df = pd.read_csv('dataset/recipes_small.csv' ,encoding='latin-1',on_bad_lines='skip')

df.head()

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,crab filled crescent snacks,94947,70,111448,2004-07-03,"['time-to-make', 'course', 'main-ingredient', ...","[69.2, 3.0, 9.0, 6.0, 5.0, 4.0, 3.0]",16,"['heat over to 375 degrees', 'spray large cook...",found in a crescent roll recipe magazine.,"['crabmeat', 'cream cheese', 'green onions', '...",9
1,curried bean salad,429010,20,300249,2010-06-08,"['curries', '30-minutes-or-less', 'time-to-mak...","[256.0, 2.0, 40.0, 18.0, 18.0, 1.0, 18.0]",4,"['drain & rinse beans', 'stir all ingredients ...",serve this flavorful and refreshing salad as a...,"['garbanzo beans', 'black beans', 'onion', 'gi...",12
2,delicious steak with onion marinade,277542,25,234062,2008-01-08,"['lactose', '30-minutes-or-less', 'time-to-mak...","[58.6, 5.0, 19.0, 0.0, 0.0, 2.0, 2.0]",6,['heat the oil in a heavy-based pan and cook t...,"another i've not tried, but looks good! times ...","['olive oil', 'red onion', 'light brown sugar'...",5
3,pork tenderloin with hoisin,78450,15,42651,2003-12-10,"['15-minutes-or-less', 'time-to-make', 'course...","[241.5, 12.0, 20.0, 45.0, 62.0, 13.0, 2.0]",7,"['cut pork into 1 / 4-inch slices', 'in a larg...",another keeper from bonnie stern's heartsmart ...,"['pork tenderloin', 'soy sauce', 'hoisin sauce...",10
4,mixed baby greens with oranges grapefruit and...,80012,15,1533,2004-01-01,"['15-minutes-or-less', 'time-to-make', 'course...","[212.8, 24.0, 30.0, 0.0, 4.0, 11.0, 6.0]",2,['in a salad bowl combine the lettuce with the...,i love grapefruit in a salad and this one is p...,"['mixed baby greens', 'oranges', 'grapefruit',...",8


In [65]:
# nombre total de ligne

len(df)

500

In [36]:
# affichage des colonnes

df.columns

Index(['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'ingredients',
       'n_ingredients'],
      dtype='str')

In [37]:
# type de chaque donnee

df.dtypes

name                str
id                int64
minutes           int64
contributor_id    int64
submitted           str
tags                str
nutrition           str
n_steps           int64
steps               str
description         str
ingredients         str
n_ingredients     int64
dtype: object

In [38]:
# verifier les valeurs manquantes

df.isnull().sum()

name              0
id                0
minutes           0
contributor_id    0
submitted         0
tags              0
nutrition         0
n_steps           0
steps             0
description       9
ingredients       0
n_ingredients     0
dtype: int64

In [41]:
df =df.head(3000)

In [42]:
# supprimer la ligne de la recette manquante

df =df.dropna(subset=['name'])

In [43]:
# suppression de la colonne description

df =df.drop(columns=['description'])

In [44]:
# colonnes utiles

df =df[['name' , 'ingredients' ,'minutes' ,'tags']]

In [45]:
# nettoyer les crochets et guillemets dans les ingredients

df['ingredients'] = df['ingredients'].apply(lambda x: x.replace('[','').replace(']','').replace("'",""))

In [46]:
# nettoyer les tags

df['tags'] =df['tags'].apply(lambda x: x.replace('[','').replace(']','').replace("'",""))

In [47]:
df.head()

,name,ingredients,minutes,tags
0,crab filled crescent snacks,"crabmeat, cream cheese, green onions, garlic s...",70,"time-to-make, course, main-ingredient, prepara..."
1,curried bean salad,"garbanzo beans, black beans, onion, ginger pas...",20,"curries, 30-minutes-or-less, time-to-make, cou..."
2,delicious steak with onion marinade,"olive oil, red onion, light brown sugar, balsa...",25,"lactose, 30-minutes-or-less, time-to-make, cou..."
3,pork tenderloin with hoisin,"pork tenderloin, soy sauce, hoisin sauce, rice...",15,"15-minutes-or-less, time-to-make, course, main..."
4,mixed baby greens with oranges grapefruit and...,"mixed baby greens, oranges, grapefruit, avocad...",15,"15-minutes-or-less, time-to-make, course, main..."


In [48]:
# remplacer les valeurs manquantes par une chaine vide

df['ingredients'] =df['ingredients'].fillna('')
df['tags'] =df['tags'].fillna('')

In [49]:
# fusion des tags et ingredients pour la comparaison

df['text'] =df['ingredients'] + " " + df['tags']

In [50]:
# mettre en miniscule et suppression des caracters speciaux

df['text'] = df['text'].str.lower()
df['text'] = df['text'].str.replace('[^a-zA-Z]', '', regex=True)

In [51]:
# transformation du texte en vecteur

from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(stop_words='english')
tfidf_matrix=vectorizer.fit_transform(df['text'])

In [52]:
# calcul de la simiarite entre les recettes et association de chaque recette a son index

from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(tfidf_matrix,tfidf_matrix)
indices= pd.Series(df.index,index=df['name']).drop_duplicates()

In [55]:
# fonction de recommendation

def recommander(nom_recette, max_minutes=60, top_n=5):
    idx= indices[nom_recette]
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = scores[1:top_n+1]
    recette_indices = [i[0] for i in scores]
    similarites= [i[1] for i in scores]
    resultats=df[['name', 'ingredients', 'minutes', 'tags']].iloc[recette_indices]
    resultats=resultats[resultats['minutes'] <= max_minutes]   
    resultats['score']=similarites[:len(resultats)]
    
    return resultats

In [56]:
df['name'].head(10)

0                          crab filled crescent snacks
1                                   curried bean salad
2                  delicious steak with onion marinade
3                          pork tenderloin with hoisin
4    mixed baby greens with oranges  grapefruit and...
5                                 buttercream frosting
6              creamy cheesy scrambled eggs with basil
7                                 christmas hard candy
8                                  pilgrim s cole slaw
9              creme de menthe chocolate vanilla drink
Name: name, dtype: str

In [58]:
# exemple de recommendation

recommander("crab filled crescent snacks", max_minutes=60)

,name,ingredients,minutes,tags,score
1,curried bean salad,"garbanzo beans, black beans, onion, ginger pas...",20,"curries, 30-minutes-or-less, time-to-make, cou...",0.0
2,delicious steak with onion marinade,"olive oil, red onion, light brown sugar, balsa...",25,"lactose, 30-minutes-or-less, time-to-make, cou...",0.0
3,pork tenderloin with hoisin,"pork tenderloin, soy sauce, hoisin sauce, rice...",15,"15-minutes-or-less, time-to-make, course, main...",0.0
4,mixed baby greens with oranges grapefruit and...,"mixed baby greens, oranges, grapefruit, avocad...",15,"15-minutes-or-less, time-to-make, course, main...",0.0
5,buttercream frosting,"sugar, all-purpose flour, salt, milk, unsalted...",30,"30-minutes-or-less, time-to-make, course, prep...",0.0


In [59]:
recommander("crab filled crescent snacks", max_minutes=30)

,name,ingredients,minutes,tags,score
1,curried bean salad,"garbanzo beans, black beans, onion, ginger pas...",20,"curries, 30-minutes-or-less, time-to-make, cou...",0.0
2,delicious steak with onion marinade,"olive oil, red onion, light brown sugar, balsa...",25,"lactose, 30-minutes-or-less, time-to-make, cou...",0.0
3,pork tenderloin with hoisin,"pork tenderloin, soy sauce, hoisin sauce, rice...",15,"15-minutes-or-less, time-to-make, course, main...",0.0
4,mixed baby greens with oranges grapefruit and...,"mixed baby greens, oranges, grapefruit, avocad...",15,"15-minutes-or-less, time-to-make, course, main...",0.0
5,buttercream frosting,"sugar, all-purpose flour, salt, milk, unsalted...",30,"30-minutes-or-less, time-to-make, course, prep...",0.0
